In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
from pathlib import Path
from matplotlib import rc

In [ ]:
#rc("font", **{"family": "serif", "serif": ["Times"], "size": 15})
rc("text", usetex=False)


In [ ]:
result_example = json.loads(Path("jp_draytek_vigorap_912c_v1.4.13__0cf916d3b7f1ea5691d10bafbf5814abfecb21cd67a3f037b8d5e5acec2528df_fs_0.json").read_text())

In [ ]:
elfs = {
    "elf": [],
    "juiciness": [],
    "Alpine": [],
    "Buildroot": [],
    "OpenEmbedded": [],
    "OpenWrt": [],
    "PTXdist": [],
    "Ubuntu": [],
    "NSRL": [],
    "Manpages": [],
    "Strings": [],
    "Filename": [],
    "Taint Symbols": [],
}

mapping = {
    "keyword_in_filename": "Filename",
    "keyword_in_binary": "Strings",
    "known_taint_symbol": "Taint Symbols",
    "file_not_in_nsrl": "NSRL",
    "not_in_ubuntu_manpages": "Manpages",
    "confidence_file_not_in_alpine_repos": "Alpine",
    "confidence_file_not_in_buildroot": "Buildroot",
    "confidence_file_not_in_openwrt_repos": "OpenWrt",
    "confidence_file_not_in_ptxdist_distrokit": "PTXdist",
    "confidence_file_not_in_ubuntu_repos": "Ubuntu",
    "confidence_file_not_in_openembedded_layers": "OpenEmbedded",
}

for elf in sorted(result_example["results"], key=lambda x: x["juiciness"], reverse=False):
    elfs["elf"] += [elf["name"]]
    elfs["juiciness"] += [elf["juiciness"]]

    for k, v in elf["ranking_by_factor"].items():
        elfs[mapping[k]] += [v["result"]]

In [ ]:
df = pd.DataFrame(elfs).sort_values("juiciness", ascending=False).reset_index().drop(columns=["index"])
df

In [ ]:
df["juicy_buckets"] = df["juiciness"].round(0).astype(int)

nice_examples = [
    "dray_fwup", "dray_lic_chk",
    "upload_settings.cgi", "goahead",
    "upload_speed.cgi", "dray_traffic",
    "flash_data", "libwifisoncfg.so",
    "libuClibc++.so.0", "wlanconfig",
    "udevtrigger", "libuci.so",
    "busybox", "librt-0.9.33.2.so",
    "dropbear", "hostapd",
    "iperf3", "wpa_supplicant",
    "libstdc++.so.6", "strace",
    "libz.so.1.2.7", "libz.so"
]
df[df.isin(nice_examples)]


sampled_df = df[df["elf"].isin(nice_examples)].groupby("juicy_buckets").head(2).sort_values("juiciness").drop(columns=["juiciness", "juicy_buckets"])


In [ ]:
palette = ["#8A9A5B"] + sns.color_palette("colorblind")[::-1]

ax = sampled_df.plot(kind='barh', edgecolor="black",stacked=True, color=palette, figsize=(8, 5))
ax.legend(ncols=4, bbox_to_anchor=(1.05, 1.3))

ax.set_axisbelow(True)
ax.grid()
ax.set_xticks(range(0, 12))
ax.set_yticklabels(sampled_df['elf'])
ax.set_xlabel("Juiciness by Factor")
ax.set_ylabel("Sampled ELF Targets")
ax.invert_xaxis()

ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(ax.get_yticks())
ax2.set_yticklabels(sampled_df.index)
ax2.set_ylabel("JuicePress Order")

handles, labels = ax.get_legend_handles_labels()

group_guessers = [0, 1, 2, 3, 4, 5][::-1]
group_misc = [6, 7][::-1]
group_kw = [8, 9, 10][::-1]


ax.add_artist(ax.legend(
    [handles[i] for i in group_guessers],
    [labels[i] for i in group_guessers],
    title="Package Guessers",
    loc="upper right",
    ncol=2,
    bbox_to_anchor=(1.10, 1.375),
))

ax.add_artist(ax.legend(
    [handles[i] for i in group_misc],
    [labels[i] for i in group_misc],
    title="Misc.",
    loc="upper center",
    bbox_to_anchor=(0.245, 1.375),
))


# Second legend (Group B)
ax.legend(
    [handles[i] for i in group_kw],
    [labels[i] for i in group_kw],
    title="Keyword Matchers",
    loc="upper left",
    bbox_to_anchor=(-0.3, 1.375),
)

plt.savefig("example_juicepress_order_draytek_vigorap_912c_v1.4.13.pdf", bbox_inches="tight")

plt.show()